# Brain Tumor MRI Classification with Neural Networks and Deep Learning

**Course:** MLAI 504 – Neural Networks and Deep Learning (2026–2027)  
**Author:** karam jammal & yorgo bou fayad 
**Dataset:** [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) (Kaggle, CC0: Public Domain)

## Project goal

Build a neural network that classifies a brain MRI slice into one of four classes: **glioma**, **meningioma**, **pituitary tumor** or **no tumor**. The project follows a systematic approach: data exploration and cleaning, step-by-step model creation, hyperparameter tuning and transfer learning, with every decision explained.

> ⚠️ **Disclaimer:** this is an academic project. The model is not a medical device and must not be used for diagnosis.

## Table of contents

0. Environment setup
1. Dataset selection and description
2. Data exploration and preprocessing
3. Step-by-step model creation
4. Hyperparameter tuning
5. Transfer learning
6. Integration of course topics
7. Evaluation and interpretation
8. Final results and conclusions
9. References

---
# 0. Environment setup

The notebook runs in a Python virtual environment with TensorFlow / Keras. This section imports the libraries, fixes the random seeds and records the library versions so the results can be reproduced.

In [1]:
import os
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import cv2
import imagehash
import kagglehub
import tensorflow as tf
import keras
import keras_tuner as kt

# Reproducibility: one call fixes the Python, NumPy and TensorFlow random seeds
SEED = 42
keras.utils.set_random_seed(SEED)

# Record the environment
versions = {
    "Python": platform.python_version(),
    "TensorFlow": tf.__version__,
    "Keras": keras.__version__,
    "Keras backend": keras.backend.backend(),
    "Keras Tuner": kt.__version__,
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "OpenCV": cv2.__version__,
    "ImageHash": imagehash.__version__,
    "kagglehub": kagglehub.__version__,
}
for name, version in versions.items():
    print(f"{name:<14}: {version}")

# GPU check (0 is expected on a Windows laptop; the GPU comes in Phase 3 via Colab)
gpus = tf.config.list_physical_devices("GPU")
print(f"\nGPUs available: {len(gpus)}")

c:\Users\hp\Desktop\miai504_Project\brain-tumor-mri-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python        : 3.12.4
TensorFlow    : 2.21.0
Keras         : 3.15.1
Keras backend : tensorflow
Keras Tuner   : 1.4.8
NumPy         : 2.5.4
Pandas        : 3.1.1
scikit-learn  : 1.9.1
OpenCV        : 5.0.0
ImageHash     : 4.3.2
kagglehub     : 1.0.3

GPUs available: 0


**Download the dataset.** `kagglehub` downloads the dataset from Kaggle the first time and reuses the local copy afterwards. The folder tree below confirms the data is in place and gives a first image count per class.

In [2]:
# Download the dataset (only the first time; later runs reuse the local copy)
DATA_DIR = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Dataset location:", DATA_DIR, "\n")

# Print the folder tree with the number of images in each folder
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")
for root, dirs, files in sorted(os.walk(DATA_DIR)):
    depth = root[len(DATA_DIR):].count(os.sep)
    n_images = sum(f.lower().endswith(IMAGE_EXTENSIONS) for f in files)
    label = f"  ({n_images} images)" if n_images else ""
    print(f"{'    ' * depth}{os.path.basename(root)}/{label}")

100%|██████████| 157M/157M [08:23<00:00, 327kB/s] 

Extracting files...


Dataset location: C:\Users\hp\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\2 

2/
    Testing/
        glioma/  (400 images)
        meningioma/  (400 images)
        notumor/  (400 images)
        pituitary/  (400 images)
    Training/
        glioma/  (1400 images)
        meningioma/  (1400 images)
        notumor/  (1400 images)
        pituitary/  (1400 images)
